# Run experiment on Colab — RT-SWT-nhom2 (LLM Unit Test Generation)

This notebook runs the **Python / LLM part** on Colab: clone repo -> install -> generate tests with `gpt-4o-mini`.

**Java measurement (JaCoCo/PIT via Maven) must run on a LOCAL machine** — Colab cannot build Java. Here we only: (1) generate tests with the LLM, (2) measure the Python side, (3) run the analysis notebooks.

Run the cells top-to-bottom.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Nguyen-Tung-An/RT-SWT-nhom2.git
%cd RT-SWT-nhom2

# >>> BRANCH = nhanh chua code thi nghiem. Doi lai neu can (vd 'main' sau khi merge) <<<
BRANCH = 'locTX-1002'
!git checkout $BRANCH
!git pull   # get the latest commit

## 2. Install dependencies

In [ ]:
!pip install openai pandas numpy scipy matplotlib coverage lizard -q
print('deps installed')

## 3. Mount Drive + point DATA_ROOT at the source dataset

`run_experiment.py` reads each function's source code from `DATA_ROOT/<raw_source_path>` (e.g. `java_functions/JA-002.java`). Upload the `research/` folder (the one containing `java_functions/`, `python_functions/`, `raw/`) to your Drive, then set `DATA_ROOT` below to that folder.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# >>> EDIT THIS to the folder that contains java_functions/ python_functions/ raw/ <<<
os.environ['DATA_ROOT'] = '/content/drive/MyDrive/RBL-experiment/research/research'

print('DATA_ROOT =', os.environ['DATA_ROOT'])
assert os.path.isdir(os.environ['DATA_ROOT']), 'DATA_ROOT not found - fix the path above'
!ls "$DATA_ROOT"

## 4. OpenAI API key
Add a Colab **Secret** named `openai` (key icon in the left sidebar), value = your `sk-...` key, and enable *Notebook access*.

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('openai')
print('key length:', len(os.environ['OPENAI_API_KEY']))

## 5. Gate E3 — API smoke test

In [ ]:
!python scripts/test_api.py

## 6. Generate tests (LLM)
Default input = `data/pilot_sample.csv` (pilot). For the **full** run, set `DATASET_CSV` to the full functions CSV before running (uncomment below).

In [ ]:
# import os; os.environ['DATASET_CSV'] = 'data/functions.csv'   # <-- for the FULL run
!python scripts/run_experiment.py

## 7. Check the output

In [ ]:
print(open('results/pilot_api_log.txt').read()[-1500:])
import pandas as pd
df = pd.read_csv('results/pilot_llm_output.csv')
print('\nrows:', len(df), '| columns:', list(df.columns))
df.head()

## 8. Commit results back to GitHub (optional)
Download `results/` and `generated_tests/` and commit them from your local machine, **or** configure git here with a personal access token. Never commit the API key.